# Практика · Діагностика: що робити далі

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.md](homework.md) · Тест: [quiz.html](quiz.html)

Дошка оголошень про вживані телефони має сама відсіювати шахрайські оголошення.
Перша модель уже навчена й дала якесь число. Уся ця практика — про те, як із цього
числа вивести наступний крок.

Що зробимо:

1. Порахуємо **три базові лінії**: дурну константу, просте правило й людину-модератора.
2. Оцінимо **стелю задачі** двома способами — через узгодженість модераторів і через оракула.
3. Навчимо першу модель і поставимо діагноз за деревом рішень із лекції.
4. Побудуємо **криві навчання** трьох видів і прочитаємо їх.
5. Візьмемо **сто помилок**, згрупуємо за причиною й порахуємо ціну кожної групи в гривнях.
6. Додамо знайдену ознаку й чесно заміряємо приріст — а потім ще три ознаки, які не дадуть нічого.
7. Підженемо **степеневий закон** до кривої «дані → якість» і дізнаємось, скільки треба даних.
8. Складемо **журнал експериментів** таблицею.


## 1 · Дошка оголошень

Оголошення бувають трьох категорій, і ціни в них зовсім різні: телефон коштує тисячі,
ноутбук — десятки тисяч, аксесуар — сотні. Схильність до шахрайства залежить від того,
наскільки ціна занижена **всередині своєї категорії**, від віку акаунта, кількості фото
й довжини опису.

Мітку не вигадуємо руками: для кожного оголошення рахуємо справжню ймовірність шахрайства
й кидаємо монетку з цією ймовірністю. Тому в задачі є чесна випадковість — і, як наслідок,
є стеля, вище якої не стрибне ніхто.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_ads = 6000

# категорія оголошення й типова ціна в цій категорії
categories = np.array(["телефони", "ноутбуки", "аксесуари"])
category_index = rng.choice(3, n_ads, p=[0.60, 0.25, 0.15])
category = categories[category_index]
category_median = np.array([8000.0, 32000.0, 700.0])[category_index]

# ціна відносно медіани СВОЄЇ категорії — саме вона керує шахрайством
price_ratio_category = rng.lognormal(0.0, 0.25, n_ads)
price_uah = price_ratio_category * category_median * np.exp(rng.normal(0, 0.02, n_ads))

account_age_days = rng.gamma(2.0, 90.0, n_ads) + 1
photo_count = rng.poisson(4.0, n_ads)
description_length = 40 + 30 * photo_count + rng.gamma(2.0, 25.0, n_ads)

# справжня схильність до шахрайства: дешевше за свою категорію, новий акаунт, мало фото
true_logit = (2.1
              - 4.6 * (price_ratio_category - 1.0)
              - 1.0 * np.log(account_age_days / 30.0)
              - 0.18 * photo_count
              - 0.003 * description_length)
true_probability = 1 / (1 + np.exp(-true_logit))
is_fraud = (rng.random(n_ads) < true_probability).astype(int)

print("оголошень:", n_ads)
print("шахрайських:", int(is_fraud.sum()), "->", round(is_fraud.mean(), 4))


## 2 · Ознака, яка виглядає розумно, але зламана

Модель не знає про категорії. Все, що в неї є з цінового боку, — це відношення ціни
оголошення до медіанної ціни **по всій дошці**. Для телефонів це майже те саме, що треба;
для ноутбуків і аксесуарів — зовсім не те. Ми навмисно почнемо з цього набору ознак,
бо саме таку помилку і має знайти аналіз помилок.


In [ ]:
board_median_price = np.median(price_uah)
price_ratio_board = price_uah / board_median_price

# те, що бачить перша модель
features_v1 = np.column_stack([price_uah, price_ratio_board, account_age_days,
                               photo_count, description_length])
feature_names_v1 = ["price_uah", "price_ratio_board", "account_age_days",
                    "photo_count", "description_length"]

board = pd.DataFrame(features_v1, columns=feature_names_v1)
board["category"] = category
board["is_fraud"] = is_fraud

print("медіана цін по всій дошці:", round(board_median_price), "грн")
print()
print(board.groupby("category")[["price_uah", "price_ratio_board"]].median().round(2))


Ось і дефект, який ми потім знайдемо руками: у ноутбуків `price_ratio_board` близько 3.8,
у аксесуарів — близько 0.08. Модель прочитає це як «ноутбуки дуже дорогі, отже чесні»
і «аксесуари дуже дешеві, отже підозрілі», хоча ціна всередині кожної категорії
цілком звичайна.

## 3 · Ділимо дошку

Усе, що міряємо далі, міряємо на перевірочній частині.


In [ ]:
from sklearn.model_selection import train_test_split

index_all = np.arange(n_ads)
index_train, index_test = train_test_split(
    index_all, test_size=0.35, random_state=42, stratify=is_fraud)

y_train = is_fraud[index_train]
y_test = is_fraud[index_test]

print("навчальних:", len(index_train), " перевірочних:", len(index_test))
print("частка шахрайських у перевірочній:", round(y_test.mean(), 4))


## 4 · Скільки коштує кожна помилка

Заблокувати чесне оголошення (хибна тривога) коштує майданчику 200 грн, пропустити
шахрайське — 1 800 грн. Далі всі рішення ми зважуватимемо саме в гривнях, а не лише
в метриці.


In [ ]:
COST_FALSE_ALARM = 200     # заблокували чесного продавця
COST_MISS = 1800           # пропустили шахрая


def damage(y_true, y_pred):
    """Повертає (хибні тривоги, пропуски, збиток у гривнях)."""
    false_alarms = int(((y_pred == 1) & (y_true == 0)).sum())
    misses = int(((y_pred == 0) & (y_true == 1)).sum())
    return false_alarms, misses, false_alarms * COST_FALSE_ALARM + misses * COST_MISS


print("перевірка на крайньому випадку — модель, що нікого не блокує:")
print(damage(y_test, np.zeros_like(y_test)))


## 5 · Три базові лінії

Число без базової лінії не означає нічого. Рахуємо три різні, бо вони відповідають
на три різні питання: чи модель узагалі щось вивчила, чи варто було її вчити
й чи можна замінити людину.

Модератора теж треба звідкись узяти. Ми змоделюємо його чесно: людина бачить ту саму
схильність до шахрайства, що й природа, але з випадковим шумом у судженні.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

# 1. дурна константа: завжди «чесне»
dummy = DummyClassifier(strategy="most_frequent").fit(features_v1[index_train], y_train)
prediction_dummy = dummy.predict(features_v1[index_test])

# 2. просте правило, написане руками за півгодини
prediction_rule = ((price_ratio_board[index_test] < 0.9)
                   & (account_age_days[index_test] < 60)).astype(int)

# 3. модератор-людина: бачить справжню схильність, але з шумом у судженні
true_log_odds = np.log(true_probability / (1 - true_probability))
moderator_noise_1 = rng.normal(0, 0.55, n_ads)
moderator_noise_2 = rng.normal(0, 0.55, n_ads)
moderator_1 = (1 / (1 + np.exp(-(true_log_odds + moderator_noise_1))) > 0.5).astype(int)
moderator_2 = (1 / (1 + np.exp(-(true_log_odds + moderator_noise_2))) > 0.5).astype(int)

baselines = {"дурна константа": prediction_dummy,
             "просте правило": prediction_rule,
             "модератор-людина": moderator_1[index_test]}

rows = []
for name, prediction in baselines.items():
    false_alarms, misses, cost = damage(y_test, prediction)
    rows.append({"базова лінія": name,
                 "F1": round(f1_score(y_test, prediction, zero_division=0), 3),
                 "accuracy": round(accuracy_score(y_test, prediction), 3),
                 "хибних тривог": false_alarms,
                 "пропусків": misses,
                 "збиток, грн": cost})

print(pd.DataFrame(rows).to_string(index=False))


## 6 · Перевіряємо себе: рахуємо F1 руками

Перш ніж довіряти будь-якому числу, варто хоч раз порахувати його самотужки.
`precision` — частка правильних серед тих, кого модель назвала шахраями. `recall` —
частка спійманих серед усіх справжніх шахраїв. F1 — їхнє гармонійне середнє.


In [ ]:
def f1_by_hand(y_true, y_pred):
    """F1 з означення, без жодної бібліотеки — щоб побачити, що магії всередині немає."""
    true_positive = int(((y_pred == 1) & (y_true == 1)).sum())
    false_positive = int(((y_pred == 1) & (y_true == 0)).sum())
    false_negative = int(((y_pred == 0) & (y_true == 1)).sum())
    precision = true_positive / (true_positive + false_positive)
    recall = true_positive / (true_positive + false_negative)
    return 2 * precision * recall / (precision + recall)


our_f1 = f1_by_hand(y_test, moderator_1[index_test])
library_f1 = f1_score(y_test, moderator_1[index_test])

assert np.isclose(our_f1, library_f1), "розрахунок розійшовся!"
print("наш F1:      ", round(our_f1, 6))
print("бібліотечний:", round(library_f1, 6))
print("✅ збігається")


## 7 · Стеля задачі

Дно каже, чи є сенс у роботі. Стеля каже, коли її припинити. Оцінимо двома способами.

**Спосіб перший — узгодженість людей.** Даємо ті самі оголошення двом модераторам
незалежно й дивимось, як часто їхні вироки збігаються. Це орієнтир, і зазвичай
завищений: люди можуть однаково помилятись.

**Спосіб другий — оракул.** Дані ми згенерували самі, тому знаємо справжню ймовірність
шахрайства для кожного оголошення. Класифікатор, який її бачить, — найкраще, що взагалі
можливо. У житті такої розкоші немає, у навчальному прикладі є.


In [ ]:
agreement = (moderator_1[index_test] == moderator_2[index_test]).mean()
print("узгодженість двох модераторів:", round(agreement, 4))

# оракул із порогом 0.5 і оракул із найкращим для F1 порогом
oracle_half = (true_probability[index_test] > 0.5).astype(int)
thresholds = np.linspace(0.05, 0.95, 91)
oracle_f1_by_threshold = [f1_score(y_test, (true_probability[index_test] > t).astype(int))
                          for t in thresholds]
best_threshold = thresholds[int(np.argmax(oracle_f1_by_threshold))]
CEILING_F1 = max(oracle_f1_by_threshold)

print("оракул при порозі 0.50: F1 =", round(f1_score(y_test, oracle_half), 3),
      " accuracy =", round(accuracy_score(y_test, oracle_half), 3))
print("оракул при найкращому порозі", round(best_threshold, 2), ": F1 =", round(CEILING_F1, 3))
print()
print("це і є стеля задачі:", round(CEILING_F1, 3))


## 8 · Перша модель і діагноз

Тепер сама модель — звичайна логістична регресія на пʼятьох ознаках. Дивимось не на
одне число, а на три: помилку на навчанні, помилку на перевірочній і стелю.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def make_model():
    """Той самий конвеєр щоразу: масштабування + логістична регресія."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


model_v1 = make_model().fit(features_v1[index_train], y_train)
prediction_train_v1 = model_v1.predict(features_v1[index_train])
prediction_test_v1 = model_v1.predict(features_v1[index_test])
probability_test_v1 = model_v1.predict_proba(features_v1[index_test])[:, 1]

f1_train_v1 = f1_score(y_train, prediction_train_v1)
f1_test_v1 = f1_score(y_test, prediction_test_v1)
false_alarms_v1, misses_v1, cost_v1 = damage(y_test, prediction_test_v1)

print("F1 на навчанні:   ", round(f1_train_v1, 3))
print("F1 на перевірочній:", round(f1_test_v1, 3))
print("стеля задачі:      ", round(CEILING_F1, 3))
print("precision:", round(precision_score(y_test, prediction_test_v1), 3),
      " recall:", round(recall_score(y_test, prediction_test_v1), 3))
print("хибних тривог:", false_alarms_v1, " пропусків:", misses_v1,
      " збиток:", cost_v1, "грн")


Прочитаємо це деревом рішень діагноста з лекції. Напишемо його прямо кодом — так видно,
що діагноз ставиться механічно, а не на око.


In [ ]:
def diagnose(f1_train, f1_validation, ceiling, gap_threshold=0.10, ceiling_threshold=0.10):
    """Три числа на вході — одна гілка на виході."""
    gap = f1_train - f1_validation
    distance_to_ceiling = ceiling - f1_validation
    if gap > gap_threshold:
        return ("перенавчання",
                "додати даних, увімкнути регуляризацію, спростити модель")
    if distance_to_ceiling > ceiling_threshold:
        return ("недонавчання",
                "додати ознак, узяти гнучкішу модель, навчати довше")
    return ("стеля досягнута",
            "зупинитись або міняти постановку задачі")


diagnosis, treatment = diagnose(f1_train_v1, f1_test_v1, CEILING_F1)
print("розрив між навчанням і перевіркою:", round(f1_train_v1 - f1_test_v1, 3))
print("відстань до стелі:                ", round(CEILING_F1 - f1_test_v1, 3))
print()
print("діагноз:  ", diagnosis)
print("лікування:", treatment)


## 9 · Криві навчання як прилад

Крива навчання відповідає на конкретне питання: **чи допоможуть додаткові дані**.
Побудуємо три штуки — для недонавченої моделі, для перенавченої й для тієї, що вже
на стелі, — і прочитаємо кожну.


In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.tree import DecisionTreeClassifier

# повний набір ознак — з тією самою ціною, але порахованою по своїй категорії
features_v2 = np.column_stack([price_uah, price_ratio_board, account_age_days,
                               photo_count, description_length, price_ratio_category])

train_sizes = np.array([100, 200, 400, 800, 1500, 2400, 3120])

scenarios = {
    "недонавчання": (make_model(),
                     np.column_stack([account_age_days, photo_count])[index_train]),
    "перенавчання": (DecisionTreeClassifier(random_state=0), features_v2[index_train]),
    "стеля":        (make_model(), features_v2[index_train]),
}

curves = {}
for name, (estimator, features) in scenarios.items():
    sizes, score_train, score_validation = learning_curve(
        estimator, features, y_train, train_sizes=train_sizes, cv=5,
        scoring="f1", shuffle=True, random_state=0)
    curves[name] = (sizes, score_train.mean(axis=1), score_validation.mean(axis=1))
    print(name)
    print("  на навчанні: ", np.round(score_train.mean(axis=1), 3))
    print("  на валідації:", np.round(score_validation.mean(axis=1), 3))


Тепер прочитаємо форму кожної кривої тим самим кодом, яким ставили діагноз. Плюс
третя ознака — приріст за останнє подвоєння вибірки: якщо крива ще росте, дані працюють.


In [ ]:
for name, (sizes, score_train, score_validation) in curves.items():
    gap = score_train[-1] - score_validation[-1]
    last_doubling = score_validation[-1] - score_validation[-3]   # 1500 -> 3120
    diagnosis, _ = diagnose(score_train[-1], score_validation[-1], CEILING_F1)
    verdict = "дані допоможуть" if diagnosis == "перенавчання" else "дані не допоможуть"
    print(f"{name:14s} розрив {gap:+.3f}  приріст за подвоєння {last_doubling:+.3f}"
          f"  до стелі {CEILING_F1 - score_validation[-1]:.3f}  ->  {verdict}")


І та сама картинка очима.


In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for axis, (name, (sizes, score_train, score_validation)) in zip(axes, curves.items()):
    axis.plot(sizes, score_train, "o-", label="на навчанні")
    axis.plot(sizes, score_validation, "o-", label="на валідації")
    axis.axhline(CEILING_F1, linestyle="--", linewidth=1, color="gray")
    axis.set_xscale("log")
    axis.set_title(name)
    axis.set_xlabel("навчальних оголошень")
axes[0].set_ylabel("F1")
axes[0].legend()
figure.tight_layout()
plt.show()


## 10 · Сто помилок очима

Найдешевший метод у темі. Беремо рівно сто помилкових прогнозів, групуємо за причиною
й рахуємо, скільки коштує кожна група. Групи мають визначатись **тільки за тим, що видно
в даних**, — інакше в реальній задачі так зробити не вийде.


In [ ]:
error_positions = np.where(prediction_test_v1 != y_test)[0]
print("усього помилок на перевірочній:", len(error_positions), "з", len(index_test))

sample_positions = np.sort(rng.choice(error_positions, 100, replace=False))


def error_group(position):
    """Причина помилки — за спаданням підозрілості, перша ж підхожа й виграє."""
    ad = index_test[position]
    if category[ad] != "телефони":
        return "ціну порівняно не з тією медіаною"
    if 0.35 <= probability_test_v1[position] <= 0.65:
        return "модель вагалась: ймовірність біля 0.5"
    if photo_count[ad] <= 1 or account_age_days[ad] < 30:
        return "мало сигналу про продавця"
    return "модель була впевнена й помилилась"


errors = pd.DataFrame({
    "position": sample_positions,
    "group": [error_group(p) for p in sample_positions],
    "truth": y_test[sample_positions],
    "prediction": prediction_test_v1[sample_positions],
})
errors["cost"] = np.where(errors["truth"] == 1, COST_MISS, COST_FALSE_ALARM)

summary = errors.groupby("group").agg(
    помилок=("cost", "size"),
    пропусків=("truth", "sum"),
    ціна_грн=("cost", "sum")).sort_values("ціна_грн", ascending=False)
summary["хибних_тривог"] = summary["помилок"] - summary["пропусків"]
summary["частка_втрат"] = (100 * summary["ціна_грн"] / summary["ціна_грн"].sum()).round(1)

print()
print(summary[["помилок", "хибних_тривог", "пропусків", "ціна_грн", "частка_втрат"]].to_string())
print()
print("сто помилок разом коштують:", int(summary["ціна_грн"].sum()), "грн")


Одна група дає більше половини втрат. Подивимось на самі оголошення з неї — саме заради
цього метод і називається «очима».


In [ ]:
worst_group = summary.index[0]
worst_positions = errors.loc[errors["group"] == worst_group, "position"].to_numpy()[:6]
worst_ads = index_test[worst_positions]

look = pd.DataFrame({
    "категорія": category[worst_ads],
    "ціна, грн": price_uah[worst_ads].round(0),
    "до медіани дошки": price_ratio_board[worst_ads].round(2),
    "до медіани категорії": price_ratio_category[worst_ads].round(2),
    "вік акаунта": account_age_days[worst_ads].round(0),
    "фото": photo_count[worst_ads],
    "справді шахрай": y_test[worst_positions],
    "вирок моделі": prediction_test_v1[worst_positions],
})
print("група:", worst_group)
print()
print(look.to_string(index=False))


Читається з першого рядка: ціна ноутбука вчетверо вища за медіану дошки, і модель робить
із цього висновок «занадто дорого, щоб бути приманкою». А всередині своєї категорії
той самий ноутбук коштує рівно стільки, скільки має. Ознака є, але порахована не від
тієї бази.

## 11 · Одна ознака, знайдена аналізом помилок

Додаємо `price_ratio_category` — ціну відносно медіани **своєї** категорії. Жодних нових
даних, та сама модель, один новий стовпчик.


In [ ]:
model_v2 = make_model().fit(features_v2[index_train], y_train)
prediction_test_v2 = model_v2.predict(features_v2[index_test])

f1_test_v2 = f1_score(y_test, prediction_test_v2)
false_alarms_v2, misses_v2, cost_v2 = damage(y_test, prediction_test_v2)

print("F1 було:", round(f1_test_v1, 3), " стало:", round(f1_test_v2, 3),
      " приріст:", round(f1_test_v2 - f1_test_v1, 3))
print("збиток був:", cost_v1, "грн  став:", cost_v2, "грн  економія:", cost_v1 - cost_v2, "грн")
print("пропусків було:", misses_v1, " стало:", misses_v2)
print()
print("для порівняння: модератор-людина", round(f1_score(y_test, moderator_1[index_test]), 3),
      ", стеля", round(CEILING_F1, 3))


## 12 · А тепер чесна половина історії

Успіх окриляє, і хочеться додати ще ознак. Додамо три очевидні: квадрат відносної ціни,
логарифм віку акаунта й добуток кількості фото на відносну ціну. Заміряємо на
крос-валідації, бо різниця може виявитись дрібною, а один поділ на дві частини такі
дрібниці не розрізняє.


In [ ]:
from sklearn.model_selection import cross_val_score

features_v3 = np.column_stack([features_v2,
                               price_ratio_category ** 2,
                               np.log(account_age_days),
                               photo_count * price_ratio_category])

cv_v1 = cross_val_score(make_model(), features_v1[index_train], y_train, cv=5, scoring="f1")
cv_v2 = cross_val_score(make_model(), features_v2[index_train], y_train, cv=5, scoring="f1")
cv_v3 = cross_val_score(make_model(), features_v3[index_train], y_train, cv=5, scoring="f1")

for name, scores in [("5 ознак", cv_v1), ("+ ціна до категорії", cv_v2),
                     ("+ ще три похідні", cv_v3)]:
    print(f"{name:22s} F1 = {scores.mean():.3f} ± {scores.std():.3f}")

print()
print("приріст від трьох останніх ознак:", round(cv_v3.mean() - cv_v2.mean(), 4))
print("а стандартна похибка середнього тут:", round(cv_v3.std() / np.sqrt(5), 4))


Нуль. Три ознаки, жодного приросту — і це правильний результат, а не невдача практики.
Перша ознака внесла в модель **нову інформацію** (знання про категорію товару). Три інші
лише перерахували те, що модель уже могла виразити. Такі ознаки треба викидати, а не
залишати «про всяк випадок».

## 13 · Скільки дасть більше даних

Тепер оцінимо приріст **наперед**. Будуємо криву «розмір вибірки → якість» і підганяємо
до неї степеневий закон:

`F1(n) = A − B · n^(−alpha)`

де `n` — кількість навчальних оголошень, `A` — межа, до якої крива прямує при
нескінченній кількості даних, `B` і `alpha` описують, як швидко вона до цієї межі
підбирається.


In [ ]:
extrapolation_sizes = np.array([60, 100, 160, 260, 420, 680, 1100, 1800, 2600, 3120])
sizes, _, score_validation = learning_curve(
    make_model(), features_v2[index_train], y_train, train_sizes=extrapolation_sizes,
    cv=5, scoring="f1", shuffle=True, random_state=0)

quality = score_validation.mean(axis=1)
standard_error = score_validation.std(axis=1) / np.sqrt(5)

for size, value, error in zip(sizes, quality, standard_error):
    print(f"n = {size:5d}   F1 = {value:.4f} ± {error:.4f}")


Підгонка. При фіксованому `alpha` величина `n^(−alpha)` — просто число для кожної точки,
і `A` з `B` знаходяться звичайною прямою. Тому перебираємо `alpha` по сітці, а для
кожного значення розвʼязуємо задачу про пряму. Заразом переконаємось, що наша пряма
збігається з тією, яку дає `numpy`.


In [ ]:
def fit_line(x, y):
    """Пряма y = intercept + slope * x за формулами найменших квадратів."""
    n = len(x)
    slope = (n * (x * y).sum() - x.sum() * y.sum()) / (n * (x * x).sum() - x.sum() ** 2)
    intercept = (y.sum() - slope * x.sum()) / n
    return intercept, slope


def fit_power_law(sizes, quality):
    """Перебираємо alpha, для кожного шукаємо A і B, лишаємо найкращий варіант."""
    best = None
    for alpha in np.arange(0.10, 2.001, 0.01):
        transformed = sizes.astype(float) ** (-alpha)
        intercept, slope = fit_line(transformed, quality)
        residuals = intercept + slope * transformed - quality
        error = float((residuals ** 2).sum())
        if best is None or error < best[0]:
            best = (error, alpha, intercept, -slope)     # A = intercept, B = -slope
    return best[1], best[2], best[3]


# звірка: наша пряма проти numpy
check_x = sizes.astype(float) ** (-0.5)
our_intercept, our_slope = fit_line(check_x, quality)
numpy_slope, numpy_intercept = np.polyfit(check_x, quality, 1)
assert np.allclose([our_intercept, our_slope], [numpy_intercept, numpy_slope]), "розрахунок розійшовся!"
print("✅ наша пряма збігається з np.polyfit")

alpha, A, B = fit_power_law(sizes, quality)
print()
print(f"підгонка: F1(n) = {A:.4f} − {B:.3f} · n^(−{alpha:.2f})")
print("межа при нескінченних даних:", round(A, 4))
print("а зараз ми маємо:            ", round(quality[-1], 4))


Тепер головне питання: **скільки оголошень треба зібрати** для потрібного приросту.
Розвʼязуємо рівняння `A − B·n^(−alpha) = поточне + приріст` відносно `n`.


In [ ]:
def ads_needed(gain, alpha, A, B, current_quality):
    """Скільки навчальних оголошень треба, щоб піднятись на gain. None — недосяжно."""
    target = current_quality + gain
    if target >= A:
        return None
    return (B / (A - target)) ** (1 / alpha)


current = quality[-1]
current_size = sizes[-1]
print(f"зараз: {current_size} оголошень, F1 = {current:.3f}")
print()
for gain in [0.01, 0.02, 0.03, 0.04, 0.05]:
    needed = ads_needed(gain, alpha, A, B, current)
    if needed is None:
        print(f"  +{gain:.2f}  ->  недосяжно за будь-якої кількості даних")
    else:
        print(f"  +{gain:.2f}  ->  n = {needed:,.0f}".replace(",", " ")
              + f"   (×{needed / current_size:.1f})")


Та сама екстраполяція, але з коротшої ділянки. Подивимось, що сказала б підгонка,
якби ми зупинились на чотирьох перших точках, — і наскільки вона при цьому збрехала б.


In [ ]:
short_alpha, short_A, short_B = fit_power_law(sizes[:4], quality[:4])
print("підгонка по перших 4 точках: межа =", round(short_A, 3))
print("підгонка по всіх 10 точках:  межа =", round(A, 3))
print("справжня якість на повній вибірці:", round(current, 3))
print()
short_needed = ads_needed(0.02, short_alpha, short_A, short_B, quality[3])
print(f"з чотирьох точок «+0.02» вимагало б {short_needed:.0f} оголошень")
print(f"а з повної кривої — {ads_needed(0.02, alpha, A, B, current):.0f}")


Висновок, який варто запамʼятати: на короткій ділянці крива ще майже пряма, і будь-яка
екстраполяція виглядає оптимістично. Вір їй на одне-два подвоєння вперед і не вір
на порядок.

## 14 · Журнал експериментів

Одна зміна — один замір — один рядок. Складемо журнал усього, що ми сьогодні зробили,
і додамо туди два кроки, які ми не пробували в тексті: дерево без обмежень
і вагу класів.


In [ ]:
cv_tree = cross_val_score(DecisionTreeClassifier(random_state=0),
                          features_v2[index_train], y_train, cv=5, scoring="f1")
balanced_model = make_pipeline(StandardScaler(),
                               LogisticRegression(max_iter=1000, class_weight="balanced"))
cv_balanced = cross_val_score(balanced_model, features_v2[index_train], y_train,
                              cv=5, scoring="f1")

journal = pd.DataFrame([
    {"крок": 0, "що змінили": "базова лінія: одне правило",
     "F1": round(f1_score(y_test, prediction_rule), 3), "±": None,
     "вирок": "дно шкали (на тесті)"},
    {"крок": 1, "що змінили": "логістична регресія, 5 ознак",
     "F1": round(cv_v1.mean(), 3), "±": round(cv_v1.std(), 3), "вирок": "старт"},
    {"крок": 2, "що змінили": "дерево без обмеження глибини",
     "F1": round(cv_tree.mean(), 3), "±": round(cv_tree.std(), 3),
     "вирок": "на навчанні 1.000 — перенавчання"},
    {"крок": 3, "що змінили": "+ ціна до медіани категорії",
     "F1": round(cv_v2.mean(), 3), "±": round(cv_v2.std(), 3), "вирок": "прийнято"},
    {"крок": 4, "що змінили": "+ ще три похідні ознаки",
     "F1": round(cv_v3.mean(), 3), "±": round(cv_v3.std(), 3), "вирок": "відхилено"},
    {"крок": 5, "що змінили": "class_weight='balanced'",
     "F1": round(cv_balanced.mean(), 3), "±": round(cv_balanced.std(), 3),
     "вирок": "це зсув порога, не нове знання"},
])
print(journal.fillna("—").to_string(index=False))


Останній рядок варто перевірити грошима, а не метрикою: вага класів робить модель
обережнішою, і це може виявитись вигідним, навіть якщо F1 змінився мало.


In [ ]:
balanced_fitted = balanced_model.fit(features_v2[index_train], y_train)
prediction_balanced = balanced_fitted.predict(features_v2[index_test])
false_alarms_b, misses_b, cost_b = damage(y_test, prediction_balanced)

print("модель v2:            F1", round(f1_test_v2, 3),
      " хибних тривог", false_alarms_v2, " пропусків", misses_v2, " збиток", cost_v2, "грн")
print("вона ж із вагою класів: F1", round(f1_score(y_test, prediction_balanced), 3),
      " хибних тривог", false_alarms_b, " пропусків", misses_b, " збиток", cost_b, "грн")
print()
print("різниця у грошах:", cost_v2 - cost_b, "грн на кожних", len(index_test), "оголошеннях")


## 15 · Підсумок числами

Зберемо все, що ми дізнались, в одну табличку — саме її й варто показувати команді
замість фрази «модель дає 0.59».


In [ ]:
final = pd.DataFrame([
    {"позначка": "дурна константа", "F1": round(f1_score(y_test, prediction_dummy, zero_division=0), 3)},
    {"позначка": "просте правило", "F1": round(f1_score(y_test, prediction_rule), 3)},
    {"позначка": "модель v1", "F1": round(f1_test_v1, 3)},
    {"позначка": "модератор-людина", "F1": round(f1_score(y_test, moderator_1[index_test]), 3)},
    {"позначка": "модель v2", "F1": round(f1_test_v2, 3)},
    {"позначка": "стеля задачі", "F1": round(CEILING_F1, 3)},
])
print(final.to_string(index=False))
print()
print("запас, який лишився:", round(CEILING_F1 - f1_test_v2, 3), "за F1")
print("а щоб узяти з нього +0.02, потрібна навчальна вибірка n =",
      f"{ads_needed(0.02, alpha, A, B, current):,.0f}".replace(",", " "))


## Завдання

### 🟢 Рівень 1 — База

Зміни поріг для моделі v2 так, щоб збиток у гривнях став мінімальним, і порахуй,
скільки грошей це дає порівняно з порогом 0.5. Скористайся `predict_proba` й функцією
`damage`, яка вже написана вище.

**Зроблено, якщо:** названо оптимальний поріг, збиток при ньому й різниця з порогом 0.5.

### 🟡 Рівень 2 — Плюс

Повтори аналіз ста помилок для моделі **v2** — тим самим кодом, але на нових прогнозах.
Найдорожча група має змінитись. Назви її, порахуй її частку у втратах і запропонуй,
яка ознака могла б її полагодити.

**Зроблено, якщо:** є таблиця груп із цінами й одна конкретна пропозиція, що додати далі.

### 🔴 Рівень 3 — Виклик

Перевір, наскільки чесна наша екстраполяція. Візьми підгонку по перших `k` точках кривої
(для `k` від 4 до 9), спрогнозуй якість у найправішій точці й порівняй із виміряною.
Побудуй графік «скільки точок узяли → на скільки помилився прогноз».

**Зроблено, якщо:** є графік і відповідь на питання, з якої кількості точок прогноз
починає триматись у межах однієї стандартної похибки.
